# പാഠം 16 - Microsoft Foundry ഉപയോഗിച്ച് സ്കെയിലബിൾ ഏജന്റുകൾ വിന്യസിക്കൽ

ഈ നോട്ട്‌ബുക്കിൽ നിങ്ങൾ നിർമ്മിക്കുന്നത് സാങ്കൽപിക കമ്പനി **Contoso**ക്കായി **ഉത്പാദന തയാറായ കസ്റ്റമർ സപ്പോർട്ട് ഏജന്റാണ്**. മുൻപുള്ള പാഠങ്ങളിൽ നിന്ന് വ്യത്യസ്തമായി, ഇവിടെ ഏജന്റിന്റെ വിചാര ചക്രം (reasoning loop) ആണ് ലക്ഷ്യം അല്ല — ഇത് ഒരു ഏജന്റിനെ വലുതായി പ്രവർത്തിക്കാൻ സുരക്ഷിതമാക്കുന്നത് ചുറ്റിപ്പറ്റിയ എല്ലാ കാര്യങ്ങളും ആണ്:

1. **ഉപകരണം വിളിക്കൽ** — ഓർഡർ തിരയൽ, ടിക്കറ്റ് സൃഷ്ടിക്കൽ.
2. **RAG** — വിജ്ഞാനകോപനത്തിൽ നിന്നുള്ള നയപ്രകാരം ഉത്തരം.
3. **പരിശ്മരണം** — Customers ന്റെ ഇടപാടുകൾ ഓർക്കുക.
4. **മോഡൽ റൂട്ടിംഗ്** — ലളിതമായ അഭ്യർത്ഥനകൾക്ക് ചെറിയ മോഡൽ, സങ്കീർണ്ണങ്ങൾക്ക് വലിയ മോഡലിലേക്ക് അയക്കുക.
5. **ഉത്തരം കാഷിംഗ്ഗ്** — ആവർത്തിക്കുന്ന ചോദ്യങ്ങൾക്ക് മോഡൽ വിളിക്കാതെ സേവനം നൽകുക.
6. **മനുഷ്യ അംഗീകാരം** — നിശ്ചിത പരിധിയെക്കാൾ മേലുള്ള തിരിച്ചടവുകൾക്ക് ഒപ്പ് വെക്കാനുള്ള പPause.
7. **മൂല്യനിർണയ ഗേറ്റ്** — ഒരു റീലീസ് തടയാൻ ഓഫ്‌ലൈൻ ടെസ്റ്റ് സെറ്റ്.
8. **നിരീക്ഷണക്ഷമത** — ഓരോ അഭ്യർത്ഥനയുടെയും ചുറ്റും OpenTelemetry ട്രേസിംഗ്.

ഓരോ വിഭാഗവും സ്വയം പ്രവർത്തനയോഗ്യവും റണർസ്ധമായതുമാണ്. ഓരോ വരിയും വായിക്കൂ — ഉത്പാദന അടിസ്ഥാന ഘടകങ്ങൾ അറിവായി ചെറിയതാക്കിയാണ് സൂക്ഷിക്കുന്നത്.


## ക്രമീകരണം

ഈ നോട്ട്‌ബുക്ക് പ്രവർത്തിപ്പിക്കുന്നതിന് മുമ്പ്, ഉറപ്പാക്കുക നിങ്ങൾക്കുണ്ട്:

1. **പ്രതിസ്ഥാനപ്പെട്ട ചാറ്റ് മോഡൽ (ഉദാ. `gpt-5-mini`) ഉള്ള Microsoft Foundry പ്രോജക്ട്.**
2. **Azure CLI-യിൽ ലോഗിൻ ചെയ്തിരിക്കുന്നത്** — നിങ്ങളുടെ ടെർമിനലിൽ `az login` റൺ ചെയ്യുക.
3. **ആവശ്യമായ പരിസര വേരിയബിൾസ് ക്രമീകരിച്ചിരിക്കണം:**
   - `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Microsoft Foundry പ്രോജക്ട് എൻഡ്‌പോയിന്റ്.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങൾ പ്രയോഗിച്ച മോഡൽ നാമം.

RAG വിഭാഗം `AZURE_SEARCH_SERVICE_ENDPOINT`യും `AZURE_SEARCH_API_KEY`യും ക്രമീകരിച്ചാൽ **Azure AI Search** ഉപയോഗിക്കുന്നു, ഇല്ലെങ്കിൽ നോട്ട്‌ബുക്ക് സേർച്ച് റിസോഴ്‌സ് ഇല്ലാതെ പ്രവർത്തിക്കാൻ ഇൻ-മെമ്മറി സേർച്ച് പിന്തുണയ്ക്കുന്നു.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. ഉപകരണങ്ങൾ

പ്രൊഡക്ഷൻ ഉപകരണങ്ങൾ യാഥാർത്ഥ്യ സിസ്റ്റങ്ങളോടു നേരിട്ടുള്ള പ്രവർത്തനം നടത്തുന്നു. ഇവിടെ നാം ഒരു ഓർഡർ ഡാറ്റാബേസ് ഉം ടിക്കറ്റ് സിസ്റ്റം ഉം സമാനമായി സാധാരണ Python ഫങ്ഷനുകളോടെ അനുകരിക്കുന്നു. `@tool` ഡെക്കറേറ്റർ അവയെ ഏജന്റിന് അനുകൂലമാക്കുന്നു.

ഒരു കൃത്യരേഖ മീതിയുള്ള പണം തിരിച്ചടവുകൾക്കായി `issue_refund` `approval_mode="always_require"` ഉപയോഗിക്കുന്നതായി ശ്രദ്ധിക്കുക — ഇത് നാം പിന്നീട് വിന്യസിക്കുന്ന മനുഷ്യസംദേശത്തിൽ പുരോഗമനത്തിന് ആവശ്യമുള്ള പ്രാഥമിക ഘടകമാണ്.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — നയം ജ്ഞാനം അടിസ്ഥാനക്കാഴ്ച

നയത്തെക്കുറിച്ചുള്ള ചോദ്യങ്ങൾക്ക് ("നിങ്ങളുടെ റിട്ടേൺ വിൻഡോ എന്താണ്?") മോഡലിന്റെ ഓർമ്മയിൽ നിന്ന് അല്ല, അധികാരമുള്ള ഉറവിടത്തിൽ നിന്ന് മറുപടിയുണ്ടാകണം. നാം ഒരു ചെറിയ ജ്ഞാനസമാഹാരം ഒരു തിരയൽ ഉപകരണമായി മൂടിവെയ്ക്കുന്നു.

ഉല്‍പ്പാദനത്തിൽ ഇത് **Azure AI Search** ആണ്; ഇവിടെ നോട്ട്ബുക്ക് എവിടെ വന്നാലും ഓടാൻ ഓർമ്മയിലെ കീവാക്ക്‌ഡ് തിരയൽ നൽകുന്നു, പരിതസ്ഥിതിവേരിയബിളുകൾ ലഭ്യമായപ്പോൾ സ്വയം Azure AI Search ലേക്ക് മാറുന്നു.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. മെമ്മറി

ആരോടാണ് സംസാരിക്കുന്നത് മറന്നുപോകുന്ന ഒരു പിന്തുണ ഏജന്റ് തെറ്റായ പിന്തുണ ഏജന്റാണ്. ഓരോ ഉപഭോക്താവിനും ചുരുക്കമുള്ള പ്രൊഫൈൽ സ്റ്റോർ ഞങ്ങൾ സൂക്ഷിക്കുന്നു, പിന്നീട് ഏജന്റിന്റെ നിർദേശങ്ങളിലേക്ക് ഒരു ചെറിയ സംക്ഷിപ്തം ചേർക്കുന്നു. പ്രൊഡക്ഷനിൽ ഇത് മെമ്മറി സേവനമാണ് (ലക്ഷണം 13 കാണുക); ഇവിടെ ഒരു ഡിക്ഷണറി പാറ്റേണിനെ കാണുന്നതിനുള്ളതാണ്.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. മാതൃക റൂട്ടിംഗ് மற்றும் പ്രതികരണ കാഷിംഗ്

ഒരു ഒന്നിച്ചുള്ള അഭ്യർത്ഥന കൈകാര്യം ചെയ്യുന്നാൽ രണ്ട് ചെലവ് നിയന്ത്രണങ്ങൾ:

- **റൂട്ടിംഗ്**: ഒരു ചെലവുകുറഞ്ഞ ഹ്യൂറിസ്റ്റിക് ക്ലാസിഫയർ അഭ്യർത്ഥനക്ക് ചെറിയ മോഡൽ വേണ്ടതോ വലിയ മോഡൽ വേണ്ടതോ തീരുമാനിക്കുന്നു.
- **കാഷിംഗ്**: സാധാരണവൽക്കരിച്ച പുനരാവർത്തിത ചോദ്യങ്ങൾ മോഡൽ വിളി ഇല്ലാതെ നേരിട്ട് കാഷിൽ നിന്ന് സേവനം നൽകുന്നു.

ഇവിടെ ക്ലാസിഫയർ ഉദ്ദേശിച്ചുതന്നെ ലഘുവാണ്. പ്രൊഡക്ഷനിൽ ഇത് ട്രാഫിക്കിനോട് താരതമ്യം ചെയ്ത് പരിശോധിച്ച് Foundryയുടെ Model Router ഉപയോഗിച്ച് മാറ്റി വയ്ക്കാനാകും.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. ഏജന്റ്, മനുഷ്യ അംഗീകാരം, നിരീക്ഷണ ശേഷി

ഇപ്പോൾ ഉയർന്നുവന്ന ടൂളുകളിൽ നിന്ന് ഏജന്റിനെ സംഘം ക്രമീകരിക്കുകയും ഓരോ അഭ്യർത്ഥനയിലും ഒരു OpenTelemetry സ്പാൻ ഉൾപ്പെടുത്തുകയും ചെയ്യുന്നു. `handle_support_request` ഫംഗ്ഷൻ പ്രൊഡക്ഷൻ അഭ്യർത്ഥന കൈകാര്യം ചെയ്യുകയാണ്: കണക്കേത് → മാർഗ്ഗനിർദേശിക്കൽ → ട്രേസ് → പ്രവർത്തിക്കുക → കണക്കേത്.

മനുഷ്യ അംഗീകാരമെന്നത് ഫ്രെയിംവർക്കിലൂടെ കൈകാര്യം ചെയ്യുന്നു: കാരണം `issue_refund` ന് `approval_mode="always_require"` ഉള്ളതിനാൽ, പ്രവർത്തനം ഒരുപക്ഷേ വിലക്കപ്പെടുകയും വ്യക്തമായി പരിഹരിക്കുന്ന അംഗീകാരം അഭ്യർത്ഥന പൂർത്തിയാകും.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. മൂല്യനിർണയം ഗേറ്റ്

പാഠത്തിൽ നിന്നുള്ള ഇത് റിലീസ് ഗേറ്റ് ആണ്: ഓഫ്ലൈൻ ടെസ്റ്റ് സെറ്റ് ഏജന്റിന് സ്കോർ നൽകുന്നു, പാസ്സാണ് കുറഞ്ഞ പരിധി കവിഞ്ഞാൽ മാത്രമെ വിന്യാസം മുന്നോട്ട് പോവൂ. ഇവിടെ സ്കോറർ ഒരു ലളിതമായ കീവേഡ്-ഓവർലാപ് പരിശോധനയാണ്, നോട്ട്‌ബുക്ക് സ്വയം ഉൾക്കൊള്ളാൻ; നിർമ്മാണത്തിൽ നിങ്ങൾ ഒരു LLM-അസ്-ജഡ്ജ് അല്ലെങ്കിൽ ഒരു ഫ്രെയിംവർക്ക് മൂല്യനിർണയ കാർ (പാഠം 10 കാണുക) ഉപയോഗിക്കും.


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ഒന്നിച്ച് ചേർക്കൽ: ഒരു അനുകരണ റിലീസ്

താഴെയുള്ളചെല്ല് പാഠം വിശദീകരിക്കുന്ന മുഴുവൻ ലൂപ് കാണിക്കുന്നു: മൂല്യനിർണ്ണയ ഗേറ്റ് പ്രവർത്തിപ്പിക്കുക, അത് കടന്നാൽ മാത്രമേ "പ്രവർത്തിപ്പിക്കുക" ചെയ്യുക. ഏജന്റ് പതിപ്പ് ഫൗണ്ടറി ഏജന്റ് സർവീസിലേക്ക് പ്രചരിപ്പിക്കുന്നതിന് മുമ്പ് നിങ്ങൾ CIയിൽ നടത്തേണ്ട മാതൃക ഇതാണ്.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## സംഗ്രഹം

നിങ്ങൾ ഒരു നിർമ്മാണം-സജ്ജമായ കസ്റ്റമർ പിന്തുണാ ഏജൻറ് ഒരുക്കി, എല്ലാ പ്രവർത്തന പ്രശ്നങ്ങളും അവയുമായി തമ്മിൽ ബന്ധപ്പെടുത്തി:

- **ഉപകരണങ്ങൾ, RAG, മെമ്മറി** ഏജന്റിന് കഴിവും സന്ധർഭവും നൽകുന്നു.
- **മോഡൽ റൂട്ടിംഗ് மற்றும் ക്യാഷിംഗ്** ലേറ്റൻസിയും ചെലവും നിയന്ത്രിക്കുന്നു.
- **മാനവ അംഗീകാരം** വലിയ റിഫണ്ടുകൾ പോലുള്ള ഉയർന്ന അപകടമുള്ള പ്രവർത്തനങ്ങളെ സംരക്ഷിക്കുന്നു.
- **മൂല്യനിർണയ ഗേറ്റ്** മോശം റിലീസുകൾ ഷിപ് ചെയ്യുന്നതിനു മുമ്പ് തടയുന്നു.
- **ട്രേസിംഗ്** എല്ലാ അഭ്യർത്ഥനകളും നിരീക്ഷണയോഗ്യമാക്കുന്നു.

### വെല്ലുവിളി

ഈ ഏജന്റിനെ വിപുലീകരിക്കുക:

1. **പല മോഡലുകൾക്ക് പിന്തുണ നൽകുക** — മൂന്നാമത്തെ "കാരണമാറ്റം" നിലവാരം ചേർക്കുക, ഉയർച്ചകൾ/പരാതികൾ അതിലേക്കു റൂട്ടു ചെയ്യുക.
2. **മൂല്യനിർണയ ഗേറ്റുകൾ ചേർക്കുക** — `TEST_CASES` ലേക്ക് റിഫണ്ട് അംഗീകാര സാഹചര്യങ്ങൾ ഉൾപ്പെടുത്തുക, ഗേറ്റ് പിന്‍സ്ഥിതികൾ പിടികൂടുന്നുണ്ടെന്ന് സ്ഥിരീകരിക്കുക.
3. **ചെലവിൽ ബോധമുള്ള റൂട്ടിംഗ് ചേർക്കുക** — ഓരോ അഭ്യർത്ഥനയ്ക്കും (ഇട בינൂണ്ട്, വലിയ, ക്യാഷ്) വിലകൾ പ്രവചിച്ച്, മിശ്ര ചോദ്യങ്ങളുടെ ബാച്ചിനു ശേഷം ചെലവ് റിപ്പോർട്ട് പ്രിന്റ് ചെയ്യുക.

അടുത്ത പാഠത്തിൽ നിങ്ങൾ എതിര് വശം യാത്ര ചെയ്ത് Microsoft Foundry Local, Qwen ഉപയോഗിച്ച് പൂർണ്ണമായും നിങ്ങളുടെ സ്വന്തം യന്ത്രത്തിൽ ഏജന്റിനെ இயக்கും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
